<a href="https://colab.research.google.com/github/Vishu235/bears/blob/main/colab/BEARS_T4_BDD_Run.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# BEARS T4 BDD-OIA Practical Runbook

Use this notebook in the second Google account with T4 GPU access. It is focused on making the BDD-OIA practical reconstruction presentable.

## Before Running

1. In Colab, choose `Runtime > Change runtime type > T4 GPU`.
2. In that Google account's Drive, create:

```text
MyDrive/bears_t4_data/
MyDrive/bears_t4_results/
```

3. Upload this required file:

```text
MyDrive/bears_t4_data/lastframe.zip
```

4. Optional, only if you also want HalfMNIST reproduction in this account:

```text
MyDrive/bears_t4_data/halfmnist-mnistdpl-dis-None-end.pt
```

## BDD Variants This Notebook Can Run

- `DPL-AUC`: ResNet50 features, no entropy, no concept supervision.
- `DPL-AUC + entropy`: the practical run you already completed.
- `DPL-AUC + concept supervision`: uses the BDD-OIA concept/reason labels as an upper-bound sanity check.
- `DPL-AUC + concept supervision + entropy`: optional combined variant.

The BDD numbers should be reported as a practical reconstruction, not exact paper reproduction, because the paper's `bdd_2048.zip` Faster-RCNN/CBM-AUC features are unavailable.


In [ ]:
#@title 1. Configuration
REPO_URL = "https://github.com/Vishu235/bears.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
REPO_DIR = "/content/bears"  #@param {type:"string"}

DRIVE_DATA_DIR = "/content/drive/MyDrive/bears_t4_data"  #@param {type:"string"}
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/bears_t4_results"  #@param {type:"string"}

LASTFRAME_ZIP = f"{DRIVE_DATA_DIR}/lastframe.zip"
HALFMNIST_CKPT_NAME = "halfmnist-mnistdpl-dis-None-end.pt"
HALFMNIST_CKPT_IN_DRIVE = f"{DRIVE_DATA_DIR}/{HALFMNIST_CKPT_NAME}"

# Main BDD plan. Start with 10-15 epochs if you are testing; use 30 for final tables.
RUN_BDD_PREPROCESS = True  #@param {type:"boolean"}
RUN_BDD_NO_ENTROPY = True  #@param {type:"boolean"}
RUN_BDD_ENTROPY = True  #@param {type:"boolean"}
RUN_BDD_CONCEPT_SUP = True  #@param {type:"boolean"}
RUN_BDD_CONCEPT_SUP_ENTROPY = False  #@param {type:"boolean"}

BDD_EPOCHS = 30  #@param {type:"integer"}
BDD_BATCH_SIZE = 256  #@param {type:"integer"}
BDD_FEATURE_BATCH_SIZE = 64  #@param {type:"integer"}
BDD_SEED = 42  #@param {type:"integer"}

# Optional HalfMNIST paper BEARS run. Leave false unless you want to repeat HalfMNIST on the T4 too.
RUN_HALFMNIST_PAPER_BEARS = False  #@param {type:"boolean"}
RUN_HALFMNIST_PAPER_BEARS_OOD = False  #@param {type:"boolean"}

print("Configuration ready.")


In [ ]:
#@title 2. Mount Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
#@title 3. Clone or update repository
import os
import subprocess
from pathlib import Path

repo_dir = Path(REPO_DIR)
if repo_dir.exists():
    subprocess.run(["git", "fetch", "origin"], cwd=repo_dir, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=repo_dir, check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=repo_dir, check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "log", "--oneline", "-5"], check=True)


In [ ]:
#@title 4. Install Colab-safe dependencies
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run(["python", "-m", "pip", "install", "-q", "--upgrade", "pip", "setuptools", "wheel"], check=True)
subprocess.run(["python", "-m", "pip", "install", "-q", "--prefer-binary", "-r", "requirements.colab.txt"], check=True)
print("Dependencies installed.")


In [ ]:
#@title 5. Diagnostics
import os
import subprocess

os.chdir(REPO_DIR)
subprocess.run(["python", "colab_runner.py", "--job", "diagnostics"], check=True)


In [ ]:
#@title 6. Verify uploaded data and optional checkpoint
import shutil
from pathlib import Path

repo = Path(REPO_DIR)
data_dir = Path(DRIVE_DATA_DIR)
results_dir = Path(DRIVE_RESULTS_DIR)
lastframe = Path(LASTFRAME_ZIP)

print("Drive data dir:", data_dir, "exists=", data_dir.exists())
print("Drive results dir:", results_dir, "exists=", results_dir.exists())
print("lastframe.zip:", lastframe, "exists=", lastframe.exists())
if not lastframe.exists():
    raise FileNotFoundError(f"Upload lastframe.zip to {lastframe}")

repo_ckpt = repo / "XOR_MNIST" / "data" / "ckpts" / HALFMNIST_CKPT_NAME
drive_ckpt = Path(HALFMNIST_CKPT_IN_DRIVE)
repo_ckpt.parent.mkdir(parents=True, exist_ok=True)
if drive_ckpt.exists():
    shutil.copy2(drive_ckpt, repo_ckpt)
    print("Copied HalfMNIST checkpoint:", drive_ckpt, "->", repo_ckpt)
else:
    print("Optional HalfMNIST checkpoint not found:", drive_ckpt)


In [ ]:
#@title 7. Helper for jobs
import shlex
import subprocess
from pathlib import Path

def run_job(job, *, required=True, extra_args=None):
    args = [
        "python", "colab_runner.py",
        "--job", job,
        "--lastframe-zip", LASTFRAME_ZIP,
    ]
    if extra_args:
        args.extend(str(item) for item in extra_args)

    print("\n>>> Running:", " ".join(shlex.quote(arg) for arg in args), flush=True)
    completed = subprocess.run(args, cwd=REPO_DIR, check=False)
    if completed.returncode:
        logs = sorted((Path(REPO_DIR) / "logs").glob("*.log"), key=lambda p: p.stat().st_mtime)
        if logs:
            latest = logs[-1]
            print(f"\n--- Tail of latest log: {latest} ---")
            print("".join(latest.read_text(errors="replace").splitlines(True)[-160:]))
        message = f"Job {job} failed with exit code {completed.returncode}. See logs/ for full output."
        if required:
            raise SystemExit(message)
        print("SKIPPED/FAILED:", message)
    else:
        print(f"<<< Completed {job}")
    return completed.returncode

print("Helper ready.")


In [ ]:
#@title 8. Preprocess BDD-OIA once into ResNet50 2048 features
from pathlib import Path

feature_dir = Path(REPO_DIR) / "BDD_OIA" / "data" / "bdd2048_resnet"
if RUN_BDD_PREPROCESS:
    run_job(
        "bdd_preprocess_full",
        extra_args=[
            "--feature-weights", "imagenet",
            "--feature-batch-size", str(BDD_FEATURE_BATCH_SIZE),
        ],
    )
else:
    print("Skipping preprocessing. Expected existing feature dir:", feature_dir)
    if not feature_dir.exists():
        raise FileNotFoundError(feature_dir)


In [ ]:
#@title 9. Train BDD variants
common = [
    "--epochs", str(BDD_EPOCHS),
    "--bdd-batch-size", str(BDD_BATCH_SIZE),
    "--seed", str(BDD_SEED),
]

if RUN_BDD_NO_ENTROPY:
    run_job("bdd_train_full", extra_args=common + ["--w-entropy", "0", "--h-labeled-param", "0"])
else:
    print("Skipping DPL-AUC no-entropy baseline.")

if RUN_BDD_ENTROPY:
    run_job("bdd_train_full", extra_args=common + ["--w-entropy", "1.0", "--h-labeled-param", "0"])
else:
    print("Skipping DPL-AUC + entropy.")

if RUN_BDD_CONCEPT_SUP:
    run_job("bdd_train_full", extra_args=common + ["--w-entropy", "0", "--h-labeled-param", "1.0"])
else:
    print("Skipping DPL-AUC + concept supervision.")

if RUN_BDD_CONCEPT_SUP_ENTROPY:
    run_job("bdd_train_full", extra_args=common + ["--w-entropy", "1.0", "--h-labeled-param", "1.0"])
else:
    print("Skipping DPL-AUC + concept supervision + entropy.")


In [ ]:
#@title 10. Optional HalfMNIST paper BEARS rerun on T4
from pathlib import Path

ckpt = Path(REPO_DIR) / "XOR_MNIST" / "data" / "ckpts" / HALFMNIST_CKPT_NAME
if RUN_HALFMNIST_PAPER_BEARS or RUN_HALFMNIST_PAPER_BEARS_OOD:
    if not ckpt.exists():
        raise FileNotFoundError(f"Missing optional HalfMNIST checkpoint: {ckpt}")

if RUN_HALFMNIST_PAPER_BEARS:
    run_job("halfmnist_eval", extra_args=["--eval-type", "bears", "--halfmnist-preset", "paper"])
else:
    print("Skipping HalfMNIST ID BEARS.")

if RUN_HALFMNIST_PAPER_BEARS_OOD:
    run_job("halfmnist_eval", extra_args=["--eval-type", "bears", "--halfmnist-preset", "paper", "--use-ood"])
else:
    print("Skipping HalfMNIST OOD BEARS.")


In [ ]:
#@title 11. Build BDD presentation table
import csv
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

repo = Path(REPO_DIR)
out_dir = repo / "summary_tables"
out_dir.mkdir(exist_ok=True)

def binary_ece(y_true, prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    prob = np.clip(np.asarray(prob).astype(float), 0.0, 1.0)
    y_pred = (prob >= 0.5).astype(int)
    conf = np.where(y_pred == 1, prob, 1.0 - prob)
    correct = (y_pred == y_true).astype(float)
    ece = 0.0
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (conf >= lo) & (conf <= hi) if i == 0 else (conf > lo) & (conf <= hi)
        if mask.any():
            ece += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)

def mean_binary_ece(y_true, prob, cols=None):
    if cols is None:
        cols = range(y_true.shape[1])
    return float(np.mean([binary_ece(y_true[:, i], prob[:, i]) for i in cols]))

def method_from_dir(name):
    if "csup_entropy" in name:
        return "Ours DPL-AUC + concept sup + entropy"
    if "csup" in name:
        return "Ours DPL-AUC + concept sup"
    if "entropy" in name:
        return "Ours DPL-AUC + entropy"
    return "Ours DPL-AUC"

rows = []
for csv_path in sorted((repo / "BDD_OIA" / "out" / "bdd").glob("*/test_results_of_BDD.csv")):
    if csv_path.stat().st_size == 0:
        continue
    raw = []
    with csv_path.open() as handle:
        for row in csv.reader(handle):
            vals = [float(x) for x in row if x != ""]
            if vals:
                raw.append(vals)
    if not raw:
        continue
    arr = np.asarray(raw, dtype=float)
    y_true_5 = arr[:, 0:5]
    out_8 = arr[:, 5:13]
    c_prob = arr[:, 13:34]
    c_true = arr[:, -21:]
    y_true_4 = y_true_5[:, 0:4]
    y_prob_4 = out_8[:, [1, 3, 5, 7]]
    y_pred_4 = (y_prob_4 >= 0.5).astype(int)
    c_pred = (c_prob >= 0.5).astype(int)
    fs_cols = list(range(0, 9))
    left_cols = list(range(9, 15))
    right_cols = list(range(15, 21))
    rows.append({
        "Dataset": "BDD-OIA practical reconstruction",
        "Method": method_from_dir(csv_path.parent.name),
        "Run dir": csv_path.parent.name,
        "Feature source": "ResNet50 ImageNet 2048 features from lastframe.zip",
        "Concept supervision": "Yes" if "csup" in csv_path.parent.name else "No",
        "Entropy loss": "Yes" if "entropy" in csv_path.parent.name else "No",
        "mF1(Y) 4 actions": f1_score(y_true_4, y_pred_4, average="macro", zero_division=0),
        "mF1(C)": f1_score(c_true, c_pred, average="macro", zero_division=0),
        "mECEY 4 actions": mean_binary_ece(y_true_4, y_prob_4),
        "mECEC": mean_binary_ece(c_true, c_prob),
        "ECEC(F,S)": mean_binary_ece(c_true, c_prob, fs_cols),
        "ECEC(L)": mean_binary_ece(c_true, c_prob, left_cols),
        "ECEC(R)": mean_binary_ece(c_true, c_prob, right_cols),
        "Action exact-match acc": accuracy_score(y_true_4, y_pred_4),
        "Concept exact-match acc": accuracy_score(c_true, c_pred),
        "Source file": str(csv_path),
    })

bdd_df = pd.DataFrame(rows).sort_values(["Concept supervision", "Entropy loss", "Method"])
display(bdd_df)
bdd_df.to_csv(out_dir / "bdd_variants_summary.csv", index=False)

paper_ref = pd.DataFrame([
    {"Dataset": "BDD-OIA", "Method": "Paper DPL", "Feature source": "Faster-RCNN/CBM-AUC bdd_2048.zip", "mF1(Y) 4 actions": 0.72, "mF1(C)": 0.34, "mECEY 4 actions": 0.08, "mECEC": 0.84, "ECEC(F,S)": 0.75, "ECEC(L)": 0.59, "ECEC(R)": 0.79},
    {"Dataset": "BDD-OIA", "Method": "Paper DPL + BEARS", "Feature source": "Faster-RCNN/CBM-AUC bdd_2048.zip", "mF1(Y) 4 actions": 0.70, "mF1(C)": 0.42, "mECEY 4 actions": 0.06, "mECEC": 0.58, "ECEC(F,S)": 0.14, "ECEC(L)": 0.02, "ECEC(R)": 0.10},
])
comparison = pd.concat([paper_ref, bdd_df], ignore_index=True, sort=False)
display(comparison)
comparison.to_csv(out_dir / "bdd_paper_reference_and_ours.csv", index=False)
print("Saved summaries in", out_dir)


In [ ]:
#@title 12. Save reusable artifacts to Drive
import shutil
from pathlib import Path

repo = Path(REPO_DIR)
drive = Path(DRIVE_RESULTS_DIR)
drive.mkdir(parents=True, exist_ok=True)

pairs = [
    (repo / "summary_tables", drive / "summary_tables"),
    (repo / "BDD_OIA" / "out" / "bdd", drive / "bdd_out"),
    (repo / "BDD_OIA" / "models" / "bdd", drive / "bdd_models"),
    (repo / "logs", drive / "logs"),
]
for src, dst in pairs:
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print("Copied", src, "->", dst)
    else:
        print("Missing, skipped:", src)


In [ ]:
#@title 13. Archive and copy result zip to Drive
run_job("archive_results")

import shutil
from pathlib import Path

result_dir = Path(DRIVE_RESULTS_DIR)
result_dir.mkdir(parents=True, exist_ok=True)
archives = sorted((Path(REPO_DIR) / "colab_outputs").glob("bears_results_*.zip"), key=lambda p: p.stat().st_mtime)
if not archives:
    raise FileNotFoundError("No result archive found in colab_outputs.")
latest = archives[-1]
target = result_dir / latest.name
shutil.copy2(latest, target)
print("Copied", latest, "to", target)


In [ ]:
#@title 14. Optional browser download zip
import subprocess
from pathlib import Path

zip_items = [
    "bears/summary_tables",
    "bears/BDD_OIA/out",
    "bears/BDD_OIA/models/bdd",
    "bears/logs",
    "bears/colab/BEARS_T4_BDD_Run.ipynb",
    "bears/colab_runner.py",
    "bears/requirements.colab.txt",
    "bears/README.md",
]
existing = [item for item in zip_items if (Path("/content") / item).exists()]
subprocess.run(["zip", "-r", "bears_t4_bdd_results.zip", *existing], cwd="/content", check=True)
subprocess.run(["ls", "-lh", "/content/bears_t4_bdd_results.zip"], check=True)


In [ ]:
#@title 15. Optional trigger browser download
from google.colab import files
files.download("/content/bears_t4_bdd_results.zip")
